# Forecasting and Routing Simulation for OrbisLoop

This notebook is a lightweight public proof-of-concept showing how OrbisLoop could combine volume forecasting and route-planning heuristics to improve circular supply chain operations.

## Scenario

We simulate daily material volumes across pickup locations, estimate next-day pickup demand, and use a simple nearest-neighbor routing heuristic to compare manual and optimized collection sequences.

In [ ]:
import math
import random
from statistics import mean

random.seed(42)

In [ ]:
locations = [
    {"name": "Supermarket A", "x": 2, "y": 5, "history": [18, 20, 19, 21, 20, 22, 24]},
    {"name": "Warehouse B", "x": 8, "y": 3, "history": [10, 11, 12, 12, 13, 15, 14]},
    {"name": "Store C", "x": 5, "y": 9, "history": [7, 8, 8, 9, 10, 9, 11]},
    {"name": "Market D", "x": 11, "y": 8, "history": [14, 15, 15, 16, 18, 19, 20]},
    {"name": "Processor E", "x": 14, "y": 4, "history": [5, 6, 6, 7, 7, 8, 8]}
]
depot = {"name": "Hub", "x": 0, "y": 0}

In [ ]:
def forecast_next_day(history):
    recent = history[-3:]
    trend = history[-1] - history[-3]
    return round(mean(recent) + max(trend * 0.25, 0), 1)

for location in locations:
    location['forecast'] = forecast_next_day(location['history'])

locations

In [ ]:
def distance(a, b):
    return round(math.dist((a['x'], a['y']), (b['x'], b['y'])), 2)

def route_distance(route, depot):
    total = 0
    current = depot
    for stop in route:
        total += distance(current, stop)
        current = stop
    total += distance(current, depot)
    return round(total, 2)

In [ ]:
manual_route = sorted(locations, key=lambda x: x['name'])
manual_distance = route_distance(manual_route, depot)
manual_distance

In [ ]:
def nearest_neighbor_route(stops, depot):
    remaining = stops[:]
    current = depot
    ordered = []
    while remaining:
        next_stop = min(remaining, key=lambda stop: distance(current, stop))
        ordered.append(next_stop)
        remaining.remove(next_stop)
        current = next_stop
    return ordered

optimized_route = nearest_neighbor_route(locations, depot)
optimized_distance = route_distance(optimized_route, depot)
optimized_distance

In [ ]:
summary = {
    'total_forecast_volume': round(sum(location['forecast'] for location in locations), 1),
    'manual_distance': manual_distance,
    'optimized_distance': optimized_distance,
    'distance_saved': round(manual_distance - optimized_distance, 2),
    'pickup_priority': sorted(
        [(location['name'], location['forecast']) for location in locations],
        key=lambda x: x[1],
        reverse=True
    )
}
summary

## Interpretation

This simple notebook demonstrates the type of operational intelligence OrbisLoop could expose:

- forecast likely material generation at each site
- prioritize pickups by predicted volume or urgency
- compare baseline routing with an optimized heuristic
- measure potential savings in distance and planning effort

A production implementation would add richer forecasting models, vehicle capacity constraints, traffic windows, contamination risk, processor destination logic, and historical dispatch outcomes.